# LoRA

## 1.核心思想
**LoRA 的本质：** 冻结原始的预训练模型权重，并在每个 Dense 层旁边注入可训练的“旁路”降秩矩阵（A 和 B）。微调时只更新这非常少量的参数。最终推理时，可以将旁路权重无损“合并（Merge）”回主权重中。


## 2.实现
在 PyTorch 实现中，除了保留原始冻结的线性层权重外，我们需要并排初始化两个很小的可训练矩阵 A 和 B。A 通常用 Kaiming 均匀分布或高斯分布初始化，而 B 严格初始化为零，以保证训练开始时 $W = W_0 + B A \approx W_0$。

**前向传播公式：**
给定预训练权重 $W_0 \in \mathbb{R}^{d \times k}$，输入 $x$，LoRA 修改后的输出为：
$$ h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x $$

*   $A \in \mathbb{R}^{r \times k}$：降维矩阵，通常使用随机高斯分布初始化（Kaiming Uniform）。
*   $B \in \mathbb{R}^{d \times r}$：升维矩阵，**必须初始化为全 0**，以保证初始状态下 $\Delta W = 0$，也就是微调前的输出和预训练模型完全一致。
*   $r$ (rank)：矩阵的秩，通常设置极小，如 8 或 16。
*   $\alpha$：缩放因子（Scaling Factor），用来控制 $\Delta W$ 的影响程度。

**推理时合并权重 (Merge Weights)：**
$$ W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A $$
这样在部署时，计算图里没有 A 和 B，完全没有额外的推理耗时（No Inference Latency）。



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, r: int = 8, lora_alpha: int = 16):
        super().__init__()
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.r
        
        # ==========================================
        # 主权重冻结，只让低秩旁路参与训练。
        # TODO 1: 初始化主权重和 LoRA 矩阵
        # ==========================================
        # self.linear = ???
        # self.linear.weight.requires_grad = ???
        # self.lora_A = ???
        # self.lora_B = ???
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.linear.weight.requires_grad = False
        
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, r))
        
        self.reset_parameters()

    def reset_parameters(self):
        # ==========================================
        # 主权重和 LoRA 旁路分别按各自规则初始化。
        # TODO 2: 初始化权重
        # ==========================================
        # nn.init.kaiming_uniform_(???)
        # nn.init.kaiming_uniform_(???)
        # nn.init.zeros_(???)
        nn.init.kaiming_uniform_(self.linear.weight, a=math.sqrt(5))
        
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # 先走主分支，再叠加低秩旁路的增量。
        # TODO 3: 实现前向传播
        # 1. 计算主权重的输出
        # 2. 计算 LoRA 分支的输出（先降维再升维，最后乘以缩放因子）
        # 3. 将两者相加
        # 提示: 注意矩阵转置和乘法顺序
        # ==========================================
        # result = ???
        # lora_out = ???
        result = self.linear(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling
        result += lora_out

        return result

    def merge_weights(self):
        # ==========================================
        # TODO 4: 合并权重（零延迟推理）
        # 提示: 将 LoRA 的低秩更新合并到主权重中
        # ==========================================
        # self.linear.weight.data += ???
        self.linear.weight.data += (self.lora_B @ self.lora_A) * self.scaling

In [3]:
# 运行此单元格以测试你的实现
def test_lora():
    try:
        in_dim, out_dim = 128, 256
        batch_size, seq_len = 32, 10
        layer = LoRALinear(in_dim, out_dim, r=8, lora_alpha=16)

        x = torch.randn(batch_size, seq_len, in_dim)

        # 1. 验证初始化导致 B 全零，所以初始输出等于冻结权重的输出
        with torch.no_grad():
            out_lora = layer(x)
            out_base = layer.linear(x)
            assert torch.allclose(out_lora, out_base), "初始化错误: lora_B 未被初始化为 0"

        # 2. 模拟训练一步，改变 B 的值
        layer.lora_B.data.normal_(0, 0.02)

        out_trained = layer(x)
        assert not torch.allclose(out_trained, out_base), "前向传播错误: 旁路未能注入梯度值"

        # 3. 验证合并权重的正确性
        layer.merge_weights()
        out_merged = layer.linear(x)
        assert torch.allclose(out_trained, out_merged, atol=1e-5), "权重合并错误: 合并后的输出与分离时的输出不一致！"

        print("\n✅ All Tests Passed! LoRA 核心算子实现正确。")

    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码可能未完成，导致变量未定义" if isinstance(e, NameError) else "代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"❌ 发生异常: {e}")
        raise

test_lora()


✅ All Tests Passed! LoRA 核心算子实现正确。
